<a href="https://colab.research.google.com/github/behan02/langchain-tutorial/blob/main/chapters/07-parallel_chains.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parallel Chains

In [1]:
!pip install -qU \
  langchain-core \
  langchain-google-genai \
  langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

os.environ["GOOGLE_API_KEY"] = ""

In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a movie summarizer"),
        ("human", "Please summarize the movie in brief : {movie_name}")
    ]
)

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0)

In [5]:
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

## Runnable Lambda

While LCEL often allows direct function piping, `RunnableLambda` is used to explicitly wrap custom Python functions. This ensures they are compatible with the Runnable interface, allowing them to be used seamlessly within chains while supporting features like async calls and batching.

In [7]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

def dictionary_maker(text:str)-> dict:

    return {"text" : text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

## Parallel Task 1: Chains

We can create parallel chains using a mix of standard LangChain objects and custom functions. First, we will look at **Chains**.

In LangChain, a chain is a sequence of calls to components like prompts, models, and output parsers. They are highly composable, meaning you can easily pipe the output of one component as the input to the next using the `|` operator. This allows for modular and reusable logic.

In [12]:
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a simple post for the following text for LinkedIn: {text}")])

linkedin_chain = linkedin_prompt | llm | str_parser

## Parallel Task 2: Functions

Next, we look at **Functions**.

While pre-built chains handle standard LLM workflows, custom Python functions allow you to perform arbitrary logic—such as data cleaning, API calls, or complex conditional formatting—directly within your pipeline. By wrapping these in `RunnableLambda`, they gain the same interface as any other LangChain component, making them easy to run in parallel.

In [13]:
def insta_chain(text:dict):

  text = text["text"]

  insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Instagram post generator"),
    ("human", "Create a simple post for the following text for Instagram: {text}")])

  chain_insta = insta_prompt | llm | str_parser

  result = chain_insta.invoke(text)

  return result

insta_chain_runnable = RunnableLambda(insta_chain)

## Final chain

In [14]:
final_chain = (
    prompt_template
    | llm
    | str_parser
    | dictionary_maker_runnable
    | RunnableParallel(linkedin=linkedin_chain, instagram=insta_chain_runnable)
)

In [15]:
final_chain.invoke({"movie_name":"Inception"})

{'linkedin': 'Here are a few options for a simple LinkedIn post, choose the one that best fits your style!\n\n**Option 1 (Concise & Engaging):**\n\n> What if you could plant an idea deep inside someone\'s mind? 🧠\n>\n> "Inception" follows Dom Cobb, a skilled extractor now tasked with the ultimate challenge: inception. His mission? To plant an idea, not steal one, for a chance to return home to his children.\n>\n> A mind-bending journey through dream layers, where reality blurs and personal demons threaten everything. A true masterclass in storytelling and psychological depth.\n>\n> What are your thoughts on Cobb\'s final reality? Did he make it home? 👇\n>\n> #Inception #MovieAnalysis #Storytelling #Psychology #Film\n\n**Option 2 (Slightly More Direct):**\n\n> Just revisited the incredible world of "Inception."\n>\n> The film centers on Dom Cobb, an "extractor" who steals information from dreams. Now, he\'s offered redemption: perform "inception" – planting an idea – to finally get back